# Banking Intent Inference Server

Chạy notebook này trên **Google Colab (T4 GPU)** để phục vụ intent classification cho `banking-agentic`.

**Trước khi chạy:**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Copy Pinggy public URL → paste vào `INTENT_API_URL` trong file `.env` của local project.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
%%capture
import torch, re

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')

!pip install sentencepiece protobuf 'datasets==2.21.0' 'huggingface_hub>=0.34.0'
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install fastapi uvicorn pyngrok nest_asyncio -q

## Step 3: Load Fine-tuned Model from Hugging Face

In [ ]:
import torch
import torch.nn.functional as F
from unsloth import FastModel

HF_MODEL_ID = "tunah/banking-intent"
MAX_SEQ_LENGTH = 128
NUM_LABELS = 77

ID2LABEL = {
    0: "activate_my_card", 1: "age_limit", 2: "apple_pay_or_google_pay",
    3: "atm_support", 4: "automatic_top_up",
    5: "balance_not_updated_after_bank_transfer",
    6: "balance_not_updated_after_cheque_or_cash_deposit",
    7: "beneficiary_not_allowed", 8: "cancel_transfer",
    9: "card_about_to_expire", 10: "card_acceptance", 11: "card_arrival",
    12: "card_delivery_estimate", 13: "card_linking", 14: "card_not_working",
    15: "card_payment_fee_charged", 16: "card_payment_not_recognised",
    17: "card_payment_wrong_exchange_rate", 18: "card_swallowed",
    19: "cash_withdrawal_charge", 20: "cash_withdrawal_not_recognised",
    21: "change_pin", 22: "compromised_card", 23: "contactless_not_working",
    24: "country_support", 25: "declined_card_payment",
    26: "declined_cash_withdrawal", 27: "declined_transfer",
    28: "direct_debit_payment_not_recognised", 29: "disposable_card_limits",
    30: "edit_personal_details", 31: "exchange_charge", 32: "exchange_rate",
    33: "exchange_via_app", 34: "extra_charge_on_statement",
    35: "failed_transfer", 36: "fiat_currency_support",
    37: "get_disposable_virtual_card", 38: "get_physical_card",
    39: "getting_spare_card", 40: "getting_virtual_card",
    41: "lost_or_stolen_card", 42: "lost_or_stolen_phone",
    43: "order_physical_card", 44: "passcode_forgotten",
    45: "pending_card_payment", 46: "pending_cash_withdrawal",
    47: "pending_top_up", 48: "pending_transfer", 49: "pin_blocked",
    50: "receiving_money", 51: "Refund_not_showing_up",
    52: "request_refund", 53: "reverted_card_payment?",
    54: "supported_cards_and_currencies", 55: "terminate_account",
    56: "top_up_by_bank_transfer_charge", 57: "top_up_by_card_charge",
    58: "top_up_by_cash_or_cheque", 59: "top_up_failed",
    60: "top_up_limits", 61: "top_up_reverted", 62: "topping_up_by_card",
    63: "transaction_charged_twice", 64: "transfer_fee_charged",
    65: "transfer_into_account", 66: "transfer_not_received_by_recipient",
    67: "transfer_timing", 68: "unable_to_verify_identity",
    69: "verify_my_identity", 70: "verify_source_of_funds",
    71: "verify_top_up", 72: "virtual_card_not_working",
    73: "visa_or_mastercard", 74: "why_verify_identity",
    75: "wrong_amount_of_cash_received",
    76: "wrong_exchange_rate_for_cash_withdrawal",
}

print(f"Loading model from Hugging Face: {HF_MODEL_ID} ...")
model, tokenizer = FastModel.from_pretrained(
    model_name=HF_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    num_labels=NUM_LABELS,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()

# Align classification head dtype with backbone compute dtype
compute_dtype = next(
    (p.dtype for p in model.parameters() if p.dtype in (torch.float16, torch.bfloat16)),
    torch.float16,
)
for module in model.modules():
    if hasattr(module, 'modules_to_save') and hasattr(module.modules_to_save, 'values'):
        for saved_module in module.modules_to_save.values():
            saved_module.to(compute_dtype)

print("Model loaded and ready.")

## Step 4: Define Inference Function

In [ ]:
def predict(text: str, top_k: int = 5) -> dict:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)[0]
    top_values, top_indices = torch.topk(probs, k=top_k)

    return {
        "intent": ID2LABEL[probs.argmax().item()],
        "confidence": round(probs.max().item(), 4),
        "top_k": [
            {"intent": ID2LABEL[i.item()], "score": round(s.item(), 4)}
            for i, s in zip(top_indices, top_values)
        ],
    }

# Quick smoke test
result = predict("My card was stolen, please help me block it.")
print(result)

## Step 5: Start FastAPI Server

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()

api = FastAPI(title="Banking Intent Inference Server")

class PredictRequest(BaseModel):
    text: str

@api.post("/predict")
def predict_endpoint(request: PredictRequest):
    return predict(request.text)

@api.get("/health")
def health():
    return {"status": "ok", "model": HF_MODEL_ID}

PORT = 8001

def run_server():
    uvicorn.run(api, host="0.0.0.0", port=PORT)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print(f"Server running on port {PORT}")

## Step 6: Expose via Pinggy

Copy the `https://...a.free.pinggy.link` URL and set it as `INTENT_API_URL` in your local `.env` file.

In [ ]:
import subprocess, time

pinggy_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no",
     "-p", "443", "-R", f"0:localhost:{PORT}",
     "a.pinggy.io"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

time.sleep(3)
for _ in range(20):
    line = pinggy_proc.stdout.readline().decode("utf-8", errors="ignore")
    if "pinggy.link" in line:
        print("\n=== Pinggy Public URL ===")
        print(line.strip())
        print("========================")
        print("\nPaste this URL as INTENT_API_URL in your .env file.")
        break
    print(line, end="")